# El archivo que cambia lo que ve BETO

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

| | |
|---|---|
| **Problema** | BETO recibe *Martín Fierro*. ¿Qué pierde si le damos versos aislados en lugar de estrofas? |
| **Producto** | Comparar BETO antes y después de leer estrofas; justificar una decisión. |
| **Recursos** | BETO (unos 440 MB), 200 estrofas para entrenar, 50 para medir. El entrenamiento puede tardar varios minutos en CPU. |

El archivo define qué ve el modelo tanto como sus pesos. Vamos a seguir una estrofa desde la fuente hasta la prueba final.

**¿Qué cambia al adaptar BETO con estrofas y qué no puede cambiar aunque lo entrenemos?**

> **Cómo se trabaja.** Explorá primero; después completá las líneas `COMPLETAR` de los microlaboratorios. Sus valores neutros permiten ejecutar todo el cuaderno sin resolverlos.

In [35]:
import math
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling,
    Trainer, TrainingArguments, default_data_collator, pipeline, set_seed,
)
import utils
from utils import dibujar_barras

MODELO = "dccuchile/bert-base-spanish-wwm-uncased"
SEMILLA = 42
# Fija recorte y huecos de evaluacion; no iguala todas las maquinas.
set_seed(SEMILLA)
utils.versiones()

{'torch': '2.14.1',
 'transformers': '4.57.6',
 'scikit-learn': '1.9.1',
 'pandas': '3.1.0'}

## 1. La fuente también decide el contexto

El texto es de José Hernández (*El gaucho Martín Fierro*, 1872), de dominio público. Los CSV provienen de un taller de adaptación; están documentados en `datos/LEEME.md`. El archivo termina con «pero que naides contó»: no incluye *La vuelta de Martín Fierro* (1879). No se cotejó la transcripción con una edición crítica. El poema también contiene representaciones y prejuicios de su época: que un modelo los reproduzca no los valida.

| Archivo | Una fila es | Transformación que recibimos |
|---|---|---|
| `martin_fierro.csv` | un verso | versos separados, minúsculas |
| `martin_fierro_estrofas.csv` | una estrofa | versos reunidos, con otras mayúsculas y puntuación |

**Son dos versiones del mismo material**, no dos corpus independientes. Veamos la primera fila de cada uno.

In [36]:
versos = pd.read_csv(utils.RUTA_DATOS / "martin_fierro.csv")
estrofas = pd.read_csv(utils.RUTA_DATOS / "martin_fierro_estrofas.csv")
print("Versos:", len(versos), "Estrofas:", len(estrofas))
print("\nPrimeros seis versos:")
for verso in versos["samples"].head(6):
    print(verso)
print("\nPrimera estrofa:")
print(estrofas["samples"].iloc[0])

Versos: 2316 Estrofas: 395

Primeros seis versos:
aquí me pongo a cantar
al compás de la vigüela,
que el hombre que lo desvela
una pena estraordinaria
como la ave solitaria
con el cantar se consuela.

Primera estrofa:
Aquí me pongo a cantar Al compás de la vigüela, Que el hombre que lo desvela Una pena estraordinaria Como la ave solitaria Con el cantar se consuela.


> **Para mirar.** Las seis primeras filas del archivo de versos corresponden a una sola fila del de estrofas. Al juntarlas conservamos palabras a ambos lados de un hueco, pero perdemos los saltos de verso. Tampoco podemos atribuir a BETO las diferencias de capitalización entre archivos. El taller ofrece **2.316 versos** y **395 estrofas**; entrenaremos con estrofas.

## 2. Una palabra escucha a las dos orillas

BETO es un **encoder** de la familia BERT: completa `[MASK]` usando las palabras **anteriores y posteriores**. No continúa un poema palabra por palabra como un generador. Cambiemos solo el final de una oración.

In [37]:
# fill-mask: el modelo propone palabras para el hueco [MASK]
# device=-1: corre en CPU
rellenador_base = pipeline("fill-mask", model=MODELO, device=-1)
frases = [
    "Fui al [MASK] para retirar dinero.",
    "Fui al [MASK] para ver una película.",
]
for frase in frases:
    print(frase)
    # top_k=3: las tres propuestas con mayor puntaje
    for propuesta in rellenador_base(frase, top_k=3):
        print(propuesta["token_str"], round(propuesta["score"], 3))

Device set to use cpu


Fui al [MASK] para retirar dinero.
banco 0.842
cajero 0.043
centro 0.01
Fui al [MASK] para ver una película.
cine 0.295
pueblo 0.039
parque 0.038


> **Para mirar.** El comienzo es idéntico: si cambian las propuestas, el contexto de la derecha influyó. Es un experimento sobre el mecanismo, **no** una visualización de pesos de atención. En el cuaderno 02 agregamos una cabeza de clasificación; acá la cabeza puntúa tokens que podrían ocupar un hueco.

In [38]:
frase_corta = "Aquí me pongo a [MASK]"
frase_con_estrofa = estrofas["samples"].iloc[0].replace(
    "cantar", "[MASK]", 1
)
filas = []
for tipo, frase in [
    ("verso aislado", frase_corta),
    ("estrofa", frase_con_estrofa),
]:
    propuesta = rellenador_base(frase, targets=["cantar"])[0]
    filas.append({
        "contexto": tipo,
        "probabilidad_de_cantar": propuesta["score"],
    })
utils.tabla(filas).round(3)

,contexto,probabilidad_de_cantar
0,verso aislado,0.013
1,estrofa,0.033


In [ ]:
contextos = []
puntajes = []
for fila in filas:
    contextos.append(fila["contexto"])
    puntajes.append(fila["probabilidad_de_cantar"])

dibujar_barras(
    nombres=contextos,
    valores=puntajes,
    titulo="Puntaje de BETO para cantar segun el contexto",
    formato="{:.3f}",
)

> **Para mirar.** La respuesta real del poema es `cantar`. La tabla y las barras muestran el puntaje que el mismo BETO le asigna con cada contexto. No dicen que «entienda» los versos ni que `cantar` sea su primera opción: aíslan el efecto del **contexto derecho**. Ahora tenemos una razón experimental, no decorativa, para elegir estrofas.

### Microlaboratorio 1 — Cambiar el contexto

**Tarea.** Conservá el comienzo y cambiá la actividad del final.  
**Producto.** Una tercera oración con tres propuestas.  
**Antes de empezar.** Las dos frases de arriba.

| Línea | Qué tiene que quedar | Con qué |
|---|---|---|
| (1) `mi_frase` | el comienzo idéntico y un final diferente | texto que empiece `Fui al [MASK] para ` |
| (2) `mis_propuestas` | tres propuestas | `rellenador_base` con `top_k=3` |

**Comprobación.** Verifica que cambiaste el final y pediste tres propuestas. ¿Qué palabra nueva explica la diferencia?

> **Ayuda 1.** Probá con «comprar pan».  
> **Ayuda 2.** Repetí la llamada anterior, cambiando solo el texto.

In [39]:
mi_frase = ""  # COMPLETAR (1): "Fui al [MASK] para ..."
mis_propuestas = []  # COMPLETAR (2): usar rellenador_base
mis_propuestas

[]

In [40]:
utils.comprobar(
    mi_frase.startswith("Fui al [MASK] para ") and mi_frase not in frases,
    "revisa (1): conserva el comienzo y cambia el final",
)
utils.comprobar(
    len(mis_propuestas) == 3,
    "revisa (2): pide tres propuestas al modelo base",
)

Todavía no: revisa (1): conserva el comienzo y cambia el final
Todavía no: revisa (2): pide tres propuestas al modelo base


## 3. El límite que no arregla el entrenamiento

El **tokenizador** convierte texto en números de un vocabulario fijo. BETO `uncased` no distingue mayúsculas. Una palabra puede ser un token, varias piezas con `##`, o `[UNK]` (desconocida). El poema ofrece un caso sorprendente.

In [41]:
tokenizador = AutoTokenizer.from_pretrained(MODELO)
palabras = ["vigüela", "estraordinaria", "añuda", "guitarra"]
filas = []
for palabra in palabras:
    filas.append({
        "palabra": palabra,
        "piezas": tokenizador.tokenize(palabra),
    })
utils.tabla(filas)

,palabra,piezas
0,vigüela,[[UNK]]
1,estraordinaria,"[estra, ##ordi, ##nar, ##ia]"
2,añuda,"[añ, ##uda]"
3,guitarra,[guitarra]


> **Para mirar.** `vigüela` se convierte en `[UNK]`; `estraordinaria` se parte en piezas. Adaptar cambia **pesos**, no agrega palabras al vocabulario: pedir `vigüela` como propuesta de un único `[MASK]` sería un examen imposible. No confundas la cobertura de piezas con la calidad de lo aprendido.

### Del ID al vector: dos `cantar` que no son iguales

El ID no es un vector ni una probabilidad: es la **posición en un vocabulario**. BETO consulta un vector de **768 números** para ese ID y luego combina el contexto de la oración para producir un **vector contextual**. Solo mostramos las primeras cinco coordenadas; las otras 763 no se descartan cuando el modelo trabaja.

Primero mirá los primeros doce tokens de la estrofa, con sus números. Luego compará las dos apariciones de `cantar`. ¿Obtienen el mismo ID? ¿Y el mismo vector después de mirar la estrofa?

In [42]:
primera_estrofa = estrofas["samples"].iloc[0]
numeros = tokenizador(primera_estrofa, return_tensors="pt")
ids = numeros["input_ids"][0]
filas = []
for posicion in range(12):
    id_token = int(ids[posicion])
    filas.append({
        "posicion": posicion,
        "token": tokenizador.convert_ids_to_tokens(id_token),
        "id": id_token,
    })
utils.tabla(filas)

,posicion,token,id
0,0,[CLS],4
1,1,aquí,1297
2,2,me,1094
3,3,pongo,11107
4,4,a,1012
5,5,cantar,7620
6,6,al,1074
7,7,compás,29548
8,8,de,1009
9,9,la,1032


In [43]:
import torch

modelo_base = rellenador_base.model

with torch.no_grad():
    vectores_vocabulario = modelo_base.get_input_embeddings()(
        numeros["input_ids"]
    )
    vectores_contexto = modelo_base.bert(**numeros).last_hidden_state

filas = []
for posicion in range(len(ids)):
    id_token = int(ids[posicion])
    palabra = tokenizador.convert_ids_to_tokens(id_token)
    if palabra == "cantar":
        vector_inicial = vectores_vocabulario[0, posicion, :5]
        vector_final = vectores_contexto[0, posicion, :5]
        filas.append({
            "posicion": posicion,
            "token": palabra,
            "id": id_token,
            "vector_vocabulario_5_de_768":
                vector_inicial.round(decimals=2).tolist(),
            "vector_contexto_5_de_768":
                vector_final.round(decimals=2).tolist(),
        })
utils.tabla(filas)

,posicion,token,id,vector_vocabulario_5_de_768,vector_contexto_5_de_768
0,5,cantar,7620,"[-0.0, -0.05, -0.02, 0.0, -0.04]","[0.07, -0.72, 1.32, -0.1, -0.21]"
1,31,cantar,7620,"[-0.0, -0.05, -0.02, 0.0, -0.04]","[0.14, -0.47, 1.1, 0.05, 0.73]"


> **Para mirar.** Las dos apariciones tienen **el mismo ID** y el **mismo vector de vocabulario**, pero ocupan lugares distintos: después de pasar por el encoder sus **vectores contextuales difieren**. Esto muestra qué aporta mirar una estrofa y por qué no basta con una tabla de palabras. Las cinco coordenadas son una ventana al vector, no una traducción de su significado.

In [44]:
largos = []
for estrofa in estrofas["samples"]:
    largos.append(len(tokenizador(estrofa)["input_ids"]))
MAX_TOKENS = 80
resumen_largos = pd.Series(largos).describe(percentiles=[0.5, 0.95])
print(resumen_largos[["min", "50%", "95%", "max"]])
print("Estrofas que se cortarian:", sum(pd.Series(largos) > 80))

min    10.0
50%    41.0
95%    50.0
max    75.0
dtype: float64
Estrofas que se cortarian: 0


> **Para mirar.** El largo incluye `[CLS]` y `[SEP]`. Con 80 posiciones entra **cada estrofa de este archivo**: no truncamos ninguna. Es una decisión sobre estos datos, no el límite natural de BETO. Las estrofas más cortas reciben relleno, que no cuenta como respuesta.

### Microlaboratorio 2 — Una palabra de la fuente

**Tarea.** Elegí otra palabra del poema y mirá cómo se representa.  
**Producto.** `mi_palabra` y `mis_piezas`; decidí si un solo `[MASK]` podría devolverla completa.  
**Antes de empezar.** La tabla de cuatro palabras.

| Línea | Qué tiene que quedar | Con qué |
|---|---|---|
| (1) `mi_palabra` | otra palabra de una estrofa | por ejemplo, «naides» |
| (2) `mis_piezas` | piezas del vocabulario | `tokenizador.tokenize(mi_palabra)` |

**Comprobación.** Comprueba la operación, no tu interpretación.

> **Ayuda 1.** Leé la última fila de `estrofas`.  
> **Ayuda 2.** Repetí la llamada usada en la tabla anterior.

In [45]:
mi_palabra = ""  # COMPLETAR (1): otra palabra del poema
mis_piezas = []  # COMPLETAR (2): tokenizarla
mis_piezas

[]

In [46]:
utils.comprobar(
    bool(mi_palabra.strip()) and mi_palabra not in palabras,
    "revisa (1): elegi una palabra nueva",
)
utils.comprobar(
    bool(mis_piezas) and mis_piezas == tokenizador.tokenize(mi_palabra),
    "revisa (2): tokeniza mi_palabra",
)

Todavía no: revisa (1): elegi una palabra nueva
Todavía no: revisa (2): tokeniza mi_palabra


## 4. Fabricar un ejercicio con una estrofa real

Ya vimos texto, tokens, ID y vector. Para entrenar, BETO necesita **un hueco y la respuesta original**. El `DataCollatorForLanguageModeling` elige alrededor del 15 % de los tokens: la entrada suele mostrar `[MASK]`, mientras la etiqueta conserva el ID del token original. A veces un token elegido permanece igual o se reemplaza por otro; **elegido** no significa siempre **tapado**.

Esta versión sencilla selecciona tokens, no reproduce exactamente el enmascarado de palabras completas del preentrenamiento de BETO. Primero reservamos la primera estrofa; elegimos otras **200 para entrenar**, **49 para medir** y dejamos **145 sin usar**. Son estrofas completas, no 1200 versos independientes. Para leer el mecanismo elegimos una estrofa de train con «torazo» y «cantar»; no seleccionamos una prueba por lo bien que le fue al modelo.

In [47]:
primera = estrofas["samples"].iloc[0]
mezcladas = estrofas["samples"].iloc[1:].sample(
    frac=1, random_state=SEMILLA
)
textos_train = mezcladas.iloc[:200].tolist()
textos_test = [primera]
for estrofa in mezcladas.iloc[200:249]:
    textos_test.append(estrofa)
assert not set(textos_train) & set(textos_test), (
    "Una estrofa aparece en train y test"
)
print("Train:", len(textos_train), "Test:", len(textos_test))

Train: 200 Test: 50


In [48]:
def preparar_estrofas(textos: list[str]) -> Dataset:
    """Convierte estrofas en entradas numericas de largo fijo.

    Args:
        textos: Estrofas de una misma particion.

    Returns:
        Tabla de ids y mascara de atencion.
    """
    codificados = tokenizador(
        # todas las estrofas quedan del mismo largo: se rellenan o se cortan
        textos, padding="max_length", truncation=True,
        max_length=MAX_TOKENS,
    )
    return Dataset.from_dict(dict(codificados))


datos_train = preparar_estrofas(textos_train)
datos_test = preparar_estrofas(textos_test)
recolector = DataCollatorForLanguageModeling(
    # tapa al azar el 15 % de los tokens en cada lote para que el modelo los adivine
    tokenizer=tokenizador, mlm=True, mlm_probability=0.15
)

In [49]:
set_seed(SEMILLA)
# Un ejemplo de train que contiene "torazo" y "cantar".
ejemplo = datos_train[10]
lote = recolector([ejemplo])

print("Estrofa original usada en el ejemplo:")
print(textos_train[10])
filas = []
for posicion in range(MAX_TOKENS):
    id_original = int(ejemplo["input_ids"][posicion])
    id_respuesta = int(lote["labels"][0][posicion])
    if id_respuesta != -100:
        id_entrada = int(lote["input_ids"][0][posicion])
        filas.append({
            "posicion": posicion,
            "token_original": tokenizador.convert_ids_to_tokens(
                id_original
            ),
            "id_original": id_original,
            "token_de_entrada": tokenizador.convert_ids_to_tokens(
                id_entrada
            ),
            "id_de_entrada": id_entrada,
            "id_respuesta": id_respuesta,
        })
utils.tabla(filas)

Estrofa original usada en el ejemplo:
Yo soy toro en mi rodeo Y torazo en rodeo ajeno; Siempre me tuve por güeno Y si me quieren probar, Salgan otros a cantar Y veremos quién es menos.


,posicion,token_original,id_original,token_de_entrada,id_de_entrada,id_respuesta
0,9,##az,13908,[MASK],0,13908
1,24,probar,6909,[MASK],0,6909
2,29,cantar,7620,[MASK],0,7620
3,31,veremos,7540,[MASK],0,7540


> **Para mirar.** Leé cada fila de izquierda a derecha: `token_original` → `id_original` → `token_de_entrada` → `id_de_entrada`; la `respuesta` del ejercicio conserva el ID original. Cuando `token_de_entrada` es `[MASK]`, se perdió una pista y BETO debe recuperarla mirando el resto de la estrofa. `-100` en las posiciones no seleccionadas significa «no calcular el error aquí». El número es un **índice** del vocabulario; el vector de 768 valores es otra representación, que vimos arriba.

## 5. Dos copias de BETO, un solo examen

Hasta acá usamos **BETO base**, descargado del Hub, sin entrenarlo: `rellenador_base`. Ahora cargamos **otra copia del mismo checkpoint**, `modelo_adaptado`. Antes de modificarla ambas parten de los mismos pesos. `entrenador.train()` actualiza los pesos de `modelo_adaptado` con **200 estrofas** y huecos sorteados de nuevo en cada lote. La primera copia **no cambia**.

| Nombre | De dónde sale | Qué cambia | Para qué se usa |
|---|---|---|---|
| `rellenador_base` | checkpoint de BETO | nada | propuestas originales |
| `modelo_adaptado` | otra copia del checkpoint | sus pesos, con 200 estrofas | propuestas después del ajuste |
| `prueba_fija` | 50 estrofas reservadas | ni textos ni huecos | medir ambas versiones |

Si sorteáramos otros huecos después del entrenamiento, cambiaríamos **modelo y examen** a la vez. Por eso fijamos una sola vez la prueba. No la usamos para elegir épocas ni tasa después de mirar el resultado.

In [50]:
set_seed(SEMILLA)
ejemplos_test = []
for fila in datos_test:
    ejemplos_test.append(fila)
lote_test = recolector(ejemplos_test)
prueba_fija = Dataset.from_dict({
    "input_ids": lote_test["input_ids"].tolist(),
    "attention_mask": lote_test["attention_mask"].tolist(),
    "labels": lote_test["labels"].tolist(),
})
print("Estrofas reservadas:", len(prueba_fija))
print("Tokens seleccionados:", int((lote_test["labels"] != -100).sum()))

Estrofas reservadas: 50
Tokens seleccionados: 279


In [51]:
modelo_adaptado = AutoModelForMaskedLM.from_pretrained(
    MODELO, attn_implementation="eager"
)
argumentos = TrainingArguments(
    output_dir="salida_dominio", learning_rate=2e-5,
    num_train_epochs=2, per_device_train_batch_size=8,
    per_device_eval_batch_size=8, save_strategy="no",
    # sin checkpoints, sin seguimiento externo y sin barras de progreso
    report_to="none", disable_tqdm=True,
)
entrenador = Trainer(
    model=modelo_adaptado, args=argumentos, train_dataset=datos_train,
    data_collator=recolector,
)
medidor = Trainer(
    model=modelo_adaptado, args=argumentos, eval_dataset=prueba_fija,
    data_collator=default_data_collator,
)

La **pérdida** resume el error al recuperar los tokens seleccionados. `exp(pérdida)` lo pone en otra escala para **esta prueba enmascarada**: no mide calidad literaria ni la perplejidad de un generador que predice la palabra siguiente.

In [52]:
perdida_antes = medidor.evaluate()["eval_loss"]
entrenador.train()
perdida_despues = medidor.evaluate()["eval_loss"]
comparacion = pd.DataFrame({
    "momento": ["base", "adaptado"],
    "perdida": [perdida_antes, perdida_despues],
    "exp_perdida": [math.exp(perdida_antes), math.exp(perdida_despues)],
})
comparacion.round(2)

{'eval_loss': 5.962166786193848, 'eval_model_preparation_time': 0.0033, 'eval_runtime': 0.5269, 'eval_samples_per_second': 94.889, 'eval_steps_per_second': 13.284}
{'train_runtime': 27.9745, 'train_samples_per_second': 14.299, 'train_steps_per_second': 1.787, 'train_loss': 4.869083862304688, 'epoch': 2.0}
{'eval_loss': 4.835541248321533, 'eval_model_preparation_time': 0.0033, 'eval_runtime': 0.4729, 'eval_samples_per_second': 105.727, 'eval_steps_per_second': 14.802}


,momento,perdida,exp_perdida
0,base,5.96,388.45
1,adaptado,4.84,125.91


> **Para mirar.** Usá **los valores de esta corrida**, no los del cuaderno anterior: ese entrenaba con versos aislados. Acá las dos filas usan las mismas estrofas y huecos. Si baja la pérdida, el modelo mejoró en **esta prueba sobre el mismo poema**. No demuestra que entienda la obra ni que funcione con otra edición o el español general.

### Microlaboratorio 3 — Leer el cambio

**Tarea.** Calculá cuánto cambió el error.  
**Producto.** `cambio`, una diferencia interpretable.  
**Antes de empezar.** La tabla `comparacion`.

| Línea | Qué tiene que quedar | Con qué |
|---|---|---|
| (1) `cambio` | pérdida final menos inicial | `perdida_despues` y `perdida_antes` |

**Comprobación.** Un valor negativo significa menor error en estos huecos, no en todas las palabras.

> **Ayuda 1.** Usá los valores completos, no los redondeados.  
> **Ayuda 2.** Restá el valor inicial del final.

In [53]:
cambio = 0.0  # COMPLETAR (1): perdida final menos inicial

In [54]:
utils.comprobar(
    abs(cambio - (perdida_despues - perdida_antes)) < 0.001,
    "revisa (1): resta la perdida inicial de la final",
)

Todavía no: revisa (1): resta la perdida inicial de la final


## 6. ¿Podemos proponer una palabra para un verso dañado?

Imaginá que una transcripción perdió una palabra de la primera estrofa, **reservada del entrenamiento**. La fuente permite saber que la respuesta era `cantar`, un token que BETO sí puede proponer. Veamos las tres primeras propuestas de cada copia. Después tapamos `vigüela`: esa palabra quedó como `[UNK]` y **no puede** aparecer como propuesta de un único token, aunque adaptemos los pesos.

Agregamos una oración cotidiana para recordar que el ajuste sobre una obra no evalúa el español general. La primera estrofa también pertenece a `prueba_fija`: esta escena ilustra ese resultado, no es una segunda prueba independiente.

In [55]:
rellenador_adaptado = pipeline(
    "fill-mask", model=modelo_adaptado, tokenizer=tokenizador,
    device=-1,
)
pruebas = [
    {
        "hueco": "Aquí me pongo a [MASK] al compás de la vigüela, "
        "que el hombre que lo desvela una pena estraordinaria "
        "como la ave solitaria con el cantar se consuela.",
        "en_la_fuente": "cantar",
    },
    {
        "hueco": "Aquí me pongo a cantar al compás de la [MASK], "
        "que el hombre que lo desvela una pena estraordinaria "
        "como la ave solitaria con el cantar se consuela.",
        "en_la_fuente": "vigüela",
    },
    {
        "hueco": "Fui al [MASK] para retirar dinero.",
        "en_la_fuente": "no pertenece al poema",
    },
]
filas = []
for caso in pruebas:
    base = []
    adaptado = []
    for propuesta in rellenador_base(caso["hueco"], top_k=3):
        base.append(propuesta["token_str"])
    for propuesta in rellenador_adaptado(caso["hueco"], top_k=3):
        adaptado.append(propuesta["token_str"])
    filas.append({
        "hueco": caso["hueco"],
        "en_la_fuente": caso["en_la_fuente"],
        "BETO_base": ", ".join(base),
        "BETO_adaptado": ", ".join(adaptado),
    })
utils.tabla(filas)

Device set to use mps:0


,hueco,en_la_fuente,BETO_base,BETO_adaptado
0,"Aquí me pongo a [MASK] al compás de la vigüela, que el hombre que lo desvela una pena estraordinaria como la ave solitaria con el cantar se consuela.",cantar,"escribir, decir, escuchar","cantar, pensar, llorar"
1,"Aquí me pongo a cantar al compás de la [MASK], que el hombre que lo desvela una pena estraordinaria como la ave solitaria con el cantar se consuela.",vigüela,"noche, voz, vida","vida, noche, pena"
2,Fui al [MASK] para retirar dinero.,no pertenece al poema,"banco, cajero, centro","banco, cajero, juego"


> **Para mirar.** Si el adaptado propone `cantar` entre los primeros lugares, **podría ayudar** a una persona a revisar un hueco de esa transcripción. No puede recuperar `vigüela` como un único token: el problema está en el vocabulario, no solo en los pesos. Si las propuestas cambian en la frase cotidiana, eso es una pista de posible desplazamiento, **no una medición de deterioro**. El modelo no reconstruye una edición auténtica por sí solo.

**Decisión.** Escribí cuándo usarías las propuestas como ayuda editorial, qué evidencia de `comparacion` y de esta tabla te apoya, cuándo **no** confiarías en ellas y qué otro corpus medirías.

In [56]:
mi_decision = ""  # COMPLETAR (4): uso, evidencia y limite

In [57]:
utils.comprobar(
    len(mi_decision.split()) >= 20,
    "revisa (4): apoya la decision en dos salidas y un limite",
)

Todavía no: revisa (4): apoya la decision en dos salidas y un limite


## Qué hace posible —y qué no

| Uso posible de un BERT adaptado | Condición que importa |
|---|---|
| Proponer candidatas para un hueco en una transcripción similar | Una persona compara con la fuente; no se acepta automáticamente la primera propuesta. |
| Preparar representaciones para otra tarea de este dominio | Haría falta ajustar y evaluar esa tarea por separado. |
| Continuar el poema o recuperar cualquier grafía histórica | **No:** BETO completa huecos, no genera de izquierda a derecha; `[UNK]` no se arregla solo con más entrenamiento. |

- La **unidad del archivo** decide el contexto disponible; los dos `cantar` de una estrofa comparten ID y vector de vocabulario, pero **no** vector contextual.
- La adaptación cambia pesos, **no** vocabulario. Medimos con los mismos huecos en estrofas reservadas del **mismo poema**.
- Una propuesta es una **hipótesis**, no una edición verificada ni comprensión del poema.

**Sigue:** Unsloth adapta un modelo generativo entrenando matrices pequeñas: otra arquitectura, otra tarea y otra forma de medir.

**Créditos.** Inspirado en el [taller de FAMAF-UNC / Fundación Vía Libre](https://github.com/nanom/llm_adaptation_2nlp_workshop) (Hernán Maina, Guido Ivetta, Laura Alonso Alemany y Luciana Benotti; 2023; MIT). Modelo: [BETO](https://huggingface.co/dccuchile/bert-base-spanish-wwm-uncased). Para profundizar: [adaptación de dominio en Hugging Face](https://huggingface.co/learn/llm-course/chapter7/3?fw=pt). La ficha de BETO advierte sobre las licencias del corpus de preentrenamiento.